### RQ1

In [49]:
import pandas as pd
import numpy as np
from collections import defaultdict
import math
import faiss
from sentence_transformers import SentenceTransformer
import pickle

In [ ]:
big_matrix = pd.read_csv('/root/autodl-tmp/big_matrix.csv', encoding='utf-8', engine='python')
kuairec_caption_category = pd.read_csv('/root/autodl-tmp/kuairec_caption_category.csv', encoding='utf-8', engine='python')
print(f'big_matrix columns:{big_matrix.columns.tolist()}')
print(f'kuairec_caption_category:{kuairec_caption_category.columns.tolist()}') 

big_matrix columns:['user_id', 'video_id', 'play_duration', 'video_duration', 'time', 'date', 'timestamp', 'watch_ratio']
kuairec_caption_category:['video_id', 'manual_cover_text', 'caption', 'topic_tag', 'first_level_category_id', 'first_level_category_name', 'second_level_category_id', 'second_level_category_name', 'third_level_category_id', 'third_level_category_name']


In [5]:
# 预处理
# big_matrix.isna().sum() 无缺失值
# kuairec_caption_category.isna().sum()
kuairec_caption_category = kuairec_caption_category.dropna().reset_index(drop=True) # 删除有缺失值的行
# dtype
big_matrix['video_id'] = big_matrix['video_id'].astype(str)
kuairec_caption_category['video_id'] = kuairec_caption_category['video_id'].astype(str)

Global Temporal Split

In [8]:
big_matrix = big_matrix.sort_values('timestamp').reset_index(drop=True) # 按时间升序 重置索引
n = len(big_matrix)
train_end = int(n * 0.8)
val_end = int(n * 0.9)
train = big_matrix.iloc[:train_end].copy()
val = big_matrix.iloc[train_end:val_end].copy()
test = big_matrix.iloc[val_end:].copy()
print("Train:", train.shape)
print("Val:", val.shape)
print("Test:", test.shape)

Train: (10024644, 8)
Val: (1253081, 8)
Test: (1253081, 8)


将用户划分为warm medium cold

In [9]:
user_history = train.groupby('user_id')['video_id'].nunique().sort_values() # 统计看过的不同视频数
q33 = user_history.quantile(1/3)
q67 = user_history.quantile(2/3)
def get_user_group(history_len):
    if history_len <= q33:
        return 'Cold'
    elif history_len <= q67:
        return 'Medium'
    else:
        return 'Warm'
user_profile = (user_history.rename('history_len').reset_index())
user_profile['user_group'] = (user_profile['history_len'].apply(get_user_group))
user_profile

,user_id,history_len,user_group
0,4604,72,Cold
1,3116,72,Cold
2,1303,73,Cold
3,1067,74,Cold
4,3175,82,Cold
...,...,...,...
7171,524,2747,Warm
7172,4593,2766,Warm
7173,5733,2833,Warm
7174,5412,2852,Warm


In [10]:
# 把分组信息加入 Val/Test
val = val.merge(user_profile,on='user_id',how='inner')
test = test.merge(user_profile,on='user_id',how='inner')
user_groups = dict(zip(user_profile['user_id'],user_profile['user_group']))
# 每个用户 Train 中的历史video   {user_id: {video1, video1, ...}}
user_train_items = (train.groupby('user_id')['video_id'].apply(set).to_dict()) 
# 每个用户 Test 中真正交互的video   {user_id: {video1, video1, ...}}
user_test_items = (test.groupby('user_id')['video_id'].apply(set).to_dict())

Recall@K 评估函数

In [11]:
def evaluate_recall(recommendations,user_test_items,user_groups,ks=[10, 20, 30]):
    """
    recommendations:dict{user_id: [video_id1, video_id2, ...]}每个用户的推荐列表
    user_test_items:dict{user_id: {真实交互的video_id}}
    user_groups:dict{user_id: 'Cold' / 'Medium' / 'Warm'}
    return:DataFrame[user_group/num_users/Recall@10/Recall@20/Recall@30]
    """
    results = []
    for group in ['Cold', 'Medium', 'Warm']:
        group_users = [
            user_id
            for user_id in user_test_items
            if (user_id in user_groups) and (user_groups[user_id] == group)
        ]
        if len(group_users) == 0:
            continue
        group_result = {'user_group': group,'num_users': len(group_users)}

        for k in ks:
            recalls = []
            for user_id in group_users:
                true_items = user_test_items[user_id] # dict
                rec_items = recommendations.get(user_id, [])[:k] # Top-K推荐列表
                hits = len(set(rec_items) & true_items) # 命中数量
                # Recall@K
                recall = hits / len(true_items)
                recalls.append(recall)
            group_result[f'Recall@{k}'] = np.mean(recalls) # 对该组所有用户的Recall@K取平均
        results.append(group_result)
    return pd.DataFrame(results)

覆盖率

In [12]:
def compute_coverage(recommendations, user_groups, all_items):
    """
    recommendations: {user_id: [item, ...]}
    user_groups:     {user_id: 'Cold'/'Medium'/'Warm'}
    all_items:       物品池（set 或 list）
    返回: {user_group: coverage}
    """
    all_items = set(all_items)
    if not all_items:
        return {group: 0.0 for group in ['Cold', 'Medium', 'Warm']}
    coverage = {}
    for group in ['Cold', 'Medium', 'Warm']:
        group_users = [u for u in recommendations if user_groups.get(u) == group]
        if not group_users:
            coverage[group] = 0.0
            continue
        rec_items = set()
        for u in group_users:
            rec_items.update(recommendations[u])
        coverage[group] = len(rec_items) / len(all_items)
    return coverage

##### Popular Recall

In [13]:
item_popularity = (train.groupby('video_id').size().sort_values(ascending=False)) # Train 中每个视频出现的次数
popular_items = item_popularity.index.tolist() # 按热度降序排列的视频列表

def popular_recall(user_train_items,popular_items,top_k=100):
    recommendations = {}
    for user_id, seen_items in user_train_items.items():
        recs = []
        for item_id in popular_items:
            if item_id not in seen_items: # 不推荐用户已经看过的视频
                recs.append(item_id)
            if len(recs) >= top_k: # 最多只需要top_k个
                break
        recommendations[user_id] = recs
    return recommendations

In [19]:
popular_recommendations = popular_recall(user_train_items, popular_items, top_k=200)
popular_result = evaluate_recall(recommendations=popular_recommendations, user_test_items=user_test_items,
                                 user_groups=user_groups, ks=[50,100,200])
# 物品池：用 train 里出现过的视频
all_items = set(train['video_id'].unique())
cov = compute_coverage(popular_recommendations, user_groups, all_items)
popular_result['Coverage'] = popular_result['user_group'].map(cov)
popular_result

,user_group,num_users,Recall@50,Recall@100,Recall@200,Coverage
0,Cold,2378,0.002126,0.002894,0.004632,0.050044
1,Medium,2287,0.005534,0.008984,0.015080,0.103505
2,Warm,2216,0.006121,0.010251,0.020655,0.187059


##### ItemCF

In [14]:
# 计算 Item-Item 共现相似度
def build_itemcf_similarity(train, top_k_sim=200, use_hot_penalty=True, max_user_len=500):
    user_items = train.groupby('user_id')['video_id'].apply(list).to_dict()
    item_count = defaultdict(int)
    item_co_count = defaultdict(float)

    for items in user_items.values():
        items = list(dict.fromkeys(items))[:max_user_len]  # 去重且保持顺序 截断        
        for it in items:
            item_count[it] += 1
        n = len(items)
        for i in range(n):
            ii = items[i]
            for j in range(i + 1, n):
                ij = items[j]
                # 只存单向，省一半内存
                key = (ii, ij) if ii < ij else (ij, ii)
                item_co_count[key] += 1

    item_similarity = defaultdict(dict)
    for (ii, ij), co in item_co_count.items():
        base = co / math.sqrt(item_count[ii] * item_count[ij])
        if use_hot_penalty:
            # 对"被动方"热度做惩罚（双向都要算）
            sim_ij = base / math.log(1 + item_count[ij])
            sim_ji = base / math.log(1 + item_count[ii])
        else:
            sim_ij = sim_ji = base
        item_similarity[ii][ij] = sim_ij
        item_similarity[ij][ii] = sim_ji

    # top-N 截断
    for i in list(item_similarity.keys()):
        item_similarity[i] = dict(
            sorted(item_similarity[i].items(),
                   key=lambda x: x[1], reverse=True)[:top_k_sim]
        )
    return item_similarity

item_similarity = build_itemcf_similarity(train)

def itemcf_recall(user_train_items, item_similarity, top_k=200, normalize_by_history=True):
    recommendations = {}
    for user_id, seen_items in user_train_items.items():
        seen_set = set(seen_items)
        candidate_score = defaultdict(float)
        for h in seen_items:
            sims = item_similarity.get(h)
            if not sims:
                continue
            for cand, sim in sims.items():
                if cand in seen_set:
                    continue
                candidate_score[cand] += sim

        if normalize_by_history and seen_items:
            denom = len(seen_items) ** 0.5
            for k in candidate_score:
                candidate_score[k] /= denom

        ranked = sorted(candidate_score.items(),
                        key=lambda x: x[1], reverse=True)[:top_k]
        recommendations[user_id] = [i for i, _ in ranked]
    return recommendations

In [20]:
item_similarity = build_itemcf_similarity(train, use_hot_penalty=True)
itemcf_recommendations = itemcf_recall(user_train_items, item_similarity, top_k=200)
itemcf_result = evaluate_recall(
    recommendations=itemcf_recommendations,
    user_test_items=user_test_items,user_groups=user_groups,ks=[50, 100, 200])
# 物品池
all_items = set(big_matrix['video_id'].unique())  
itemcf_coverage = compute_coverage(itemcf_recommendations, user_groups, all_items)
itemcf_result['Coverage'] = itemcf_result['user_group'].map(itemcf_coverage)
itemcf_result

,user_group,num_users,Recall@50,Recall@100,Recall@200,Coverage
0,Cold,2378,0.015218,0.021069,0.031801,0.203207
1,Medium,2287,0.008371,0.021155,0.053136,0.124720
2,Warm,2216,0.019187,0.041055,0.083724,0.095544


##### ANN

In [17]:
# 拼接视频文本
kuairec_caption_category['video_text'] = (
    '封面文案：' + kuairec_caption_category['manual_cover_text']
    + '；视频描述：' + kuairec_caption_category['caption']
    + '；话题：' + kuairec_caption_category['topic_tag']
    + '；一级类别：' + kuairec_caption_category['first_level_category_name']
    + '；二级类别：' + kuairec_caption_category['second_level_category_name']
    + '；三级类别：' + kuairec_caption_category['third_level_category_name']
)

train_items = set(train['video_id'].unique())
ann_item_df = kuairec_caption_category[kuairec_caption_category['video_id'].isin(train_items)].copy()
ann_item_df = ann_item_df.drop_duplicates('video_id').reset_index(drop=True)

# 生成 Embedding
model = SentenceTransformer('/root/local_model')
item_texts = ann_item_df['video_text'].tolist()
item_embeddings = model.encode(
    item_texts, batch_size=128, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)

# 建立 FAISS 索引
faiss_idx_to_video = dict(enumerate(ann_item_df['video_id'].tolist()))
video_to_faiss_idx = {video_id: idx for idx, video_id in faiss_idx_to_video.items()}
dimension = item_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(item_embeddings.astype(np.float32))

# 召回函数
def ann_recall(user_train_items, faiss_index, item_embeddings, 
               video_to_faiss_idx, faiss_idx_to_video, 
               top_k=200, ann_top_k=50, max_history=20):
    recommendations = {}
    for user_id, seen_items in user_train_items.items():
        seen_set = set(seen_items)
        candidate_score = defaultdict(float)
        history_items = list(seen_set)[:max_history]
        for history_item in history_items:
            if history_item not in video_to_faiss_idx:
                continue
            query_idx = video_to_faiss_idx[history_item]
            query_vector = item_embeddings[query_idx].reshape(1, -1).astype(np.float32)
            
            # FAISS 搜索
            scores, indices = faiss_index.search(query_vector, ann_top_k + 1)
            for score, idx in zip(scores[0], indices[0]):
                if idx == -1:
                    continue
                candidate_item = faiss_idx_to_video[idx]
                if candidate_item in seen_set:
                    continue
                candidate_score[candidate_item] += float(score)
                
        ranked = sorted(candidate_score.items(), key=lambda x: x[1], reverse=True)[:top_k]
        recommendations[user_id] = [item_id for item_id, score in ranked]
    return recommendations

# 执行召回
ann_recommendations = ann_recall(
    user_train_items=user_train_items,
    faiss_index=faiss_index,
    item_embeddings=item_embeddings,
    video_to_faiss_idx=video_to_faiss_idx,
    faiss_idx_to_video=faiss_idx_to_video,
    top_k=200, ann_top_k=50, max_history=50  
)

# 评估与覆盖率
ann_result = evaluate_recall(
    recommendations=ann_recommendations,
    user_test_items=user_test_items,
    user_groups=user_groups,
    ks=[50, 100, 200]
)

all_items = set(train['video_id'].unique())
ann_coverage = compute_coverage(ann_recommendations, user_groups, all_items)
ann_result['Coverage'] = ann_result['user_group'].map(ann_coverage)

ann_result

Batches: 100%|██████████| 63/63 [00:03<00:00, 19.09it/s]


,user_group,num_users,Recall@50,Recall@100,Recall@200,Coverage
0,Cold,2378,0.004987,0.009710,0.015892,0.474757
1,Medium,2287,0.005773,0.011578,0.020159,0.346561
2,Warm,2216,0.005690,0.010692,0.019542,0.251102


##### 多路召回

###### Equal-weight RRF

In [21]:
def multi_recall_rrf(popular_recommendations,itemcf_recommendations,ann_recommendations,
                    user_train_items,top_k=200,rrf_k=5):
    recommendations = {}
    all_users = (
        set(popular_recommendations.keys())
        | set(itemcf_recommendations.keys())
        | set(ann_recommendations.keys()))
    for user_id in all_users:
        seen_items = set(user_train_items.get(user_id, set()))
        candidate_score = defaultdict(float)
        # Popular
        for rank, item_id in enumerate(popular_recommendations.get(user_id, [])[:200],start=1):
            if item_id not in seen_items:
                candidate_score[item_id] += 1 / (rrf_k + rank)
        # ItemCF
        for rank, item_id in enumerate(itemcf_recommendations.get(user_id, [])[:200],start=1):
            if item_id not in seen_items:
                candidate_score[item_id] += 1 / (rrf_k + rank)
        # ANN
        for rank, item_id in enumerate(ann_recommendations.get(user_id, [])[:200],start=1):
            if item_id not in seen_items:
                candidate_score[item_id] += 1 / (rrf_k + rank)
        # RRF排序，取最终Top-200
        ranked = sorted(candidate_score.items(),key=lambda x: x[1],reverse=True)[:top_k]
        recommendations[user_id] = [item_id for item_id, score in ranked]
    return recommendations

multi_recommendations = multi_recall_rrf(
    popular_recommendations,
    itemcf_recommendations,
    ann_recommendations,
    user_train_items,
    top_k=200,
    rrf_k=5
)
multi_result = evaluate_recall(
    recommendations=multi_recommendations,
    user_test_items=user_test_items,
    user_groups=user_groups,
    ks=[50, 100, 200]
)
all_items = set(train['video_id'].unique())
multi_coverage = compute_coverage(multi_recommendations,user_groups,all_items)
multi_result['Coverage'] = (multi_result['user_group'].map(multi_coverage))
multi_result

,user_group,num_users,Recall@50,Recall@100,Recall@200,Coverage
0,Cold,2378,0.009605,0.016117,0.026131,0.437390
1,Medium,2287,0.007091,0.013295,0.028210,0.296958
2,Warm,2216,0.010412,0.020802,0.042761,0.248457


###### Weighted RRF

In [22]:
def multi_recall_weighted_rrf(popular_recommendations,itemcf_recommendations,ann_recommendations,
                              user_train_items,top_k=200,rrf_k=5,
                              popular_weight=1.0,itemcf_weight=1.0,ann_weight=1.0):
    recommendations = {}
    if popular_weight == 1 and itemcf_weight == 0 and ann_weight == 0:
        return {user_id: recs[:top_k] for user_id,recs in popular_recommendations.items()}
    if popular_weight == 0 and itemcf_weight == 1 and ann_weight == 0:
        return {user_id: recs[:top_k] for user_id,recs in itemcf_recommendations.items()}
    if popular_weight == 0 and itemcf_weight == 0 and ann_weight == 1:
        return {user_id: recs[:top_k] for user_id,recs in ann_recommendations.items()}
    all_users = (
        set(popular_recommendations.keys())
        | set(itemcf_recommendations.keys())
        | set(ann_recommendations.keys())
    )
    for user_id in all_users:
        seen_items = set(user_train_items.get(user_id,set()))
        candidate_score = defaultdict(float)
        # Popular
        for rank,item_id in enumerate(popular_recommendations.get(user_id,[])[:200],start=1):
            if item_id not in seen_items:
                candidate_score[item_id] += (popular_weight / (rrf_k + rank))
        # ItemCF
        for rank,item_id in enumerate(itemcf_recommendations.get(user_id,[])[:200],start=1):
            if item_id not in seen_items:
                candidate_score[item_id] += (itemcf_weight / (rrf_k + rank))
        # ANN
        for rank,item_id in enumerate(ann_recommendations.get(user_id,[])[:200],start=1):
            if item_id not in seen_items:
                candidate_score[item_id] += (ann_weight / (rrf_k + rank))
        ranked = sorted(candidate_score.items(),key=lambda x:x[1],reverse=True)[:top_k]
        recommendations[user_id] = [item_id for item_id,score in ranked]
    return recommendations
def run_weighted_rrf_experiments(weight_list,popular_recommendations,itemcf_recommendations,
                                  ann_recommendations,user_train_items,user_test_items,
                                  user_groups,train,top_k=200,rrf_k=5):
    all_results = []
    all_items = set(train['video_id'].unique())
    for popular_weight,itemcf_weight,ann_weight in weight_list:
        weighted_recommendations = multi_recall_weighted_rrf(
            popular_recommendations=popular_recommendations,
            itemcf_recommendations=itemcf_recommendations,
            ann_recommendations=ann_recommendations,
            user_train_items=user_train_items,
            top_k=top_k,
            rrf_k=rrf_k,
            popular_weight=popular_weight,
            itemcf_weight=itemcf_weight,
            ann_weight=ann_weight
        )
        weighted_result = evaluate_recall(
            recommendations=weighted_recommendations,
            user_test_items=user_test_items,
            user_groups=user_groups,
            ks=[50,100,200]
        )
        weighted_coverage = compute_coverage(weighted_recommendations,user_groups,all_items)
        weighted_result['Coverage'] = (weighted_result['user_group'].map(weighted_coverage))
        weighted_result['Popular_weight'] = popular_weight
        weighted_result['ItemCF_weight'] = itemcf_weight
        weighted_result['ANN_weight'] = ann_weight
        weighted_result['Weight_ratio'] = (f"{popular_weight}:{itemcf_weight}:{ann_weight}")
        all_results.append(weighted_result)
    all_results = pd.concat(all_results,ignore_index=True)
    all_results = all_results[['Weight_ratio','user_group','num_users','Recall@50','Recall@100','Recall@200','Coverage']]
    return all_results

In [23]:
weight_list = [
    (1,2,1),
    (1,3,1),
    (2,3,2)
]
weighted_all_results = run_weighted_rrf_experiments(
    weight_list,popular_recommendations,itemcf_recommendations,ann_recommendations,
    user_train_items,user_test_items,user_groups,
    train,top_k=200,rrf_k=5)
weighted_all_results

,Weight_ratio,user_group,num_users,Recall@50,Recall@100,Recall@200,Coverage
0,1:2:1,Cold,2378,0.012104,0.019552,0.027994,0.418100
1,1:2:1,Medium,2287,0.007445,0.015241,0.034991,0.279652
2,1:2:1,Warm,2216,0.012955,0.026457,0.055777,0.226190
3,1:3:1,Cold,2378,0.013428,0.020497,0.029241,0.405423
4,1:3:1,Medium,2287,0.007572,0.016353,0.039733,0.264440
5,1:3:1,Warm,2216,0.014528,0.029821,0.063038,0.207782
6,2:3:2,Cold,2378,0.011011,0.018406,0.027131,0.426146
7,2:3:2,Medium,2287,0.007237,0.014298,0.031800,0.288470
8,2:3:2,Warm,2216,0.011818,0.024225,0.050381,0.235560


In [30]:
# 评估函数
def evaluate_ranking(recommendations,test_user_items,k=10):
    ndcg_list = []
    recall_list = []
    mrr_list = []
    auc_list = []
    for user_id,true_items in test_user_items.items():
        if user_id not in recommendations:continue
        rec_items = recommendations[user_id][:k]
        # ---------- Recall@K ----------
        hits = [1 if item in true_items else 0 for item in rec_items]
        num_hits = sum(hits)
        recall = num_hits / len(true_items) if len(true_items) > 0 else 0
        recall_list.append(recall)
        # ---------- NDCG@K ----------
        dcg = sum(hit / np.log2(rank + 1) for rank,hit in enumerate(hits,1))
        ideal_hits = min(len(true_items),k)
        idcg = sum(1 / np.log2(rank + 1) for rank in range(1,ideal_hits + 1))
        ndcg = dcg / idcg if idcg > 0 else 0
        ndcg_list.append(ndcg)
        # ---------- MRR@K ----------
        rr = 0
        for rank,hit in enumerate(hits,1):
            if hit == 1:
                rr = 1 / rank
                break
        mrr_list.append(rr)
        labels = [1 if item in true_items else 0 for item in rec_items]
        if len(set(labels)) == 2:
            scores = np.arange(len(rec_items),0,-1)
            auc = roc_auc_score(labels,scores)
            auc_list.append(auc)
    return {
        'NDCG@10': np.mean(ndcg_list),
        'Recall@10': np.mean(recall_list),
        'MRR@10': np.mean(mrr_list),
        'AUC': np.mean(auc_list) if auc_list else np.nan
    }

###### 1:3:1 RRF 召回结果的top-10

In [31]:
rrf_recommendations = multi_recall_weighted_rrf(
    popular_recommendations=popular_recommendations,
    itemcf_recommendations=itemcf_recommendations,
    ann_recommendations=ann_recommendations,
    user_train_items=user_train_items,
    top_k=200,
    rrf_k=5,
    popular_weight=1,
    itemcf_weight=3,
    ann_weight=1
)
test_user_items = test.groupby('user_id')['video_id'].apply(set).to_dict()
rrf_result = evaluate_ranking(rrf_recommendations,test_user_items,k=10)
rrf_result

{'NDCG@10': np.float64(0.06005209828714811),
 'Recall@10': np.float64(0.003165343164889064),
 'MRR@10': np.float64(0.15996827011577777),
 'AUC': np.float64(0.5846548855614011)}

In [53]:
save_dir = '/root/autodl-tmp/'
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, 'rrf_recommendations.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(rrf_recommendations, f)